# 01 — Train NER (M2): PhoBERT + BIO token classification

Chạy trên Google Colab (T4/L4). Input: `data/processed/ner_dataset.jsonl` (sinh bởi `scripts/build_dataset.py`).
Output: checkpoint tại `models/ner_phobert/` (tải về máy cá nhân để chạy Tier 1 offline).

**Nhãn:** BIO cho 8 entity: PACKAGE_NAME, VALUE, FUNDING, METHOD, CONTRACT_TYPE, DURATION, INVESTOR, LOCATION.
**Metric:** entity-level F1 (seqeval), báo cáo per-entity.

In [ ]:
!pip install -q transformers datasets seqeval accelerate

In [ ]:
!git clone https://github.com/dannd/autotender-vn.git /content/autotender-vn 2>/dev/null || echo 'Repo đã tồn tại (chạy lại notebook) hoặc private — kiểm tra quyền truy cập.'
%cd /content/autotender-vn

In [ ]:
# data/processed/*.jsonl KHÔNG nằm trong git (xem .gitignore) — phải sinh lại từ
# data/samples/tender_notices.jsonl (đã commit) bằng script này.
!python scripts/build_dataset.py

In [ ]:
import json
from pathlib import Path

DATA_PATH = Path('/content/autotender-vn/data/processed/ner_dataset.jsonl')
records = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
labels = sorted({t.split('-', 1)[1] for r in records for t in r['tags'] if t != 'O'})
label_list = ['O'] + [f'{p}-{l}' for l in labels for p in ('B', 'I')]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(len(records), 'records —', label_list)

In [ ]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(records, test_size=0.2, random_state=42)
print(len(train_records), 'train /', len(val_records), 'val')

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    tokenized = tokenizer(batch['tokens'], truncation=True, is_split_into_words=True, max_length=256)
    all_labels = []
    for i, tags in enumerate(batch['tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(label2id[tags[wid]])
            else:
                label_ids.append(-100)
            prev_word = wid
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list(train_records).map(tokenize_and_align, batched=True)
val_ds = Dataset.from_list(val_records).map(tokenize_and_align, batched=True)

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    preds, labels_ = eval_pred
    preds = np.argmax(preds, axis=2)
    true_preds, true_labels = [], []
    for p, l in zip(preds, labels_):
        tp, tl = [], []
        for pi, li in zip(p, l):
            if li != -100:
                tp.append(id2label[pi])
                tl.append(id2label[li])
        true_preds.append(tp)
        true_labels.append(tl)
    print(classification_report(true_labels, true_preds))
    return {'f1': f1_score(true_labels, true_preds)}

args = TrainingArguments(
    output_dir='/content/ner_out', num_train_epochs=10, per_device_train_batch_size=8,
    per_device_eval_batch_size=8, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1', logging_steps=5, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator, compute_metrics=compute_metrics)

In [ ]:
trainer.train()
trainer.save_model('/content/models/ner_phobert')
tokenizer.save_pretrained('/content/models/ner_phobert')
print('Checkpoint saved — tải /content/models/ner_phobert về models/ner_phobert/ trong repo local để dùng Tier 1.')

## Ablation (Mục 10): PhoBERT vs XLM-R
Lặp lại các bước trên với `MODEL_NAME = 'xlm-roberta-base'` và so sánh F1 để điền vào `reports/metrics.json`.

In [ ]:
# Nen checkpoint thanh zip va tai truc tiep ve may (khong can mount Drive)
from google.colab import files
import shutil
shutil.make_archive('ner_phobert', 'zip', '/content/models/ner_phobert')
files.download('ner_phobert.zip')
print('Giai nen ner_phobert.zip vao thu muc models/ner_phobert/ trong repo local.')